In [18]:
import numpy as np
from numpy.random import rand, randint
import gurobipy as gp
from gurobipy import GRB
from sympy.utilities.iterables import multiset_permutations
from TakeTwoSetup import *

# --- Finding D and H --- #
# D: max number of days
# H: max number of scenes in a day
def find_D(n, S, durations, W):
  D = (2*n*(max(durations)+np.matrix.max(S)))/(W+np.matrix.max(S))
  if np.ceil(D)%2 == 0:
    D = int(np.floor(D))
  else:
    D=int(np.ceil(D))
  return D

D = find_D(n, S, durations, W)

def find_H(n, S, durations, W):
  increasingDurations = np.sort(durations)
  increasingChangeover = np.array(np.sort(np.matrix.flatten(S)))[0]

  vals = []
  hs = []
  for h in range(2, n+1):
    total = np.sum(increasingChangeover[:h]) + np.sum(increasingDurations[:h])
    if total <= W:
      # to make sure combination is within admissible set
      vals.append(total)
      hs.append(h)
  H = hs[np.argmax(vals)]
  return H

H = find_H(n, S, durations, W)

D, H

(9, 4)

In [19]:
setOfIdx = set([])
def finding_P(n, S, durations, W, H):
    perm = []
    for i in range(H+1):
        perm.extend(multiset_permutations(range(n), i))
    
    for p in perm:
        total = 0
        for i in range(len(p)-1):
            total += S[p[i], p[i+1]]
        for i in p:
            total += durations[i]
        if total > W:
            perm.remove(p)
    return perm

P = finding_P(n, S, durations, W, H)
k = len(P)
k, P

(1890,
 [[],
  [0],
  [1],
  [2],
  [3],
  [4],
  [5],
  [6],
  [7],
  [8],
  [0, 1],
  [0, 2],
  [0, 3],
  [0, 4],
  [0, 5],
  [0, 6],
  [0, 7],
  [0, 8],
  [1, 0],
  [1, 2],
  [1, 3],
  [1, 4],
  [1, 5],
  [1, 6],
  [1, 7],
  [1, 8],
  [2, 0],
  [2, 1],
  [2, 3],
  [2, 4],
  [2, 5],
  [2, 6],
  [2, 7],
  [2, 8],
  [3, 0],
  [3, 1],
  [3, 2],
  [3, 4],
  [3, 5],
  [3, 6],
  [3, 7],
  [3, 8],
  [4, 0],
  [4, 1],
  [4, 2],
  [4, 3],
  [4, 5],
  [4, 6],
  [4, 7],
  [4, 8],
  [5, 0],
  [5, 1],
  [5, 2],
  [5, 3],
  [5, 4],
  [5, 6],
  [5, 7],
  [5, 8],
  [6, 0],
  [6, 1],
  [6, 2],
  [6, 3],
  [6, 4],
  [6, 5],
  [6, 7],
  [6, 8],
  [7, 0],
  [7, 1],
  [7, 2],
  [7, 3],
  [7, 4],
  [7, 5],
  [7, 6],
  [7, 8],
  [8, 0],
  [8, 1],
  [8, 2],
  [8, 3],
  [8, 4],
  [8, 5],
  [8, 6],
  [8, 7],
  [0, 1, 3],
  [0, 1, 5],
  [0, 1, 7],
  [0, 2, 1],
  [0, 2, 4],
  [0, 2, 6],
  [0, 2, 8],
  [0, 3, 2],
  [0, 3, 5],
  [0, 3, 6],
  [0, 3, 8],
  [0, 4, 2],
  [0, 4, 5],
  [0, 4, 6],
  [0, 4, 8],
  [0, 5, 

In [20]:
# --- ILP_pat Model Setup --- #
ILP_pat = gp.Model("ILP_pat", env=env)

# decision variables
a = ILP_pat.addVars(k, n,vtype=GRB.BINARY, name="a")
x_pat = ILP_pat.addVars(k, D, vtype=GRB.BINARY, name="x_pat")
y_pat = ILP_pat.addVars(m, D, D, vtype=GRB.BINARY, name="y_pat")

# objective function
objFunc_pat = ILP_pat.setObjective(
    gp.quicksum(
        holdingCost[i] *
        gp.quicksum(
              (d2-d1+1)*y_pat[i, d1, d2]
              for d1 in range(D)
              for d2 in range(d1, D)
        )
        for i in range(len(actors))
    ), GRB.MINIMIZE
)

# --- constraints (lord help us) --- #
c21 = ILP_pat.addConstr(
    gp.quicksum(
        x_pat[k, 1] for k in range(len(P))
    ) == 1, name="c21"
) # Constraint (21)

c22 = ILP_pat.addConstrs((
    gp.quicksum(
        x_pat[k, d-1]
        for k in range(len(P))
        ) <=
    gp.quicksum(
        x_pat[k, d]
        for k in range(len(P))
        )
    for d in range(2, D)),
    name="c22"
) # Constraint (22)

c23 = ILP_pat.addConstrs((
    gp.quicksum(
        a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for d in range(D)
    ) == 1
    for j in range(n)),
    name="c23"
) # Constraint (23)

c24 = ILP_pat.addConstrs((
    gp.quicksum(
        y_pat[i, d1, d2]
        for d1 in range(D)
        for d2 in range(d1, D)
    ) == 1
    for i in range(m)),
    name="c24"
) # Constraint (24)

c25 = ILP_pat.addConstrs((
    gp.quicksum(
        T[i, j] * a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for j in range(n)
    ) <=
    gp.quicksum(
        y_pat[i, d1, d2]
        for d2 in range(d, D)
        for d1 in range(d)
    )
    for i in range(m) for d in range(D)),
    name="c25"
) # Constraint (25)

# --- Constraints (26-27) --- #
# These constraints are embedded in the init of the variables

In [ ]:
# --- optimize that shit --- #
ILP_pat.setAttr()
ILP_pat.optimize()
sol = ILP_pat.getJSONSolution()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 7 4700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 17 rows, 34749 columns and 28755 nonzeros (Min)


Model fingerprint: 0x10e6d6ea
Model has 405 linear objective coefficients
Model has 90 quadratic constraints
Variable types: 0 continuous, 34749 integer (34749 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [6e+02, 8e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]

Presolve added 21 rows and 0 columns
Presolve removed 0 rows and 382 columns
Presolve time: 1.20s
Presolved: 459389 rows, 187457 columns, 1870664 nonzeros
Variable types: 0 continuous, 187457 integer (187451 binary)

Deterministic concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Root barrier log...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 1.449e+04
 Factor NZ  : 7.340e+04 (roughly 1 MB of memory)
 Factor Ops : 7.177e+06 (less than 1 second per iteration)
 Threads    : 5

                  Objective   

In [ ]:
all_vars = ILP_pat.getVars()
values = ILP_pat.getAttr("X", all_vars)
names = ILP_pat.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

a[0,2] = 1.0
a[12,4] = 1.0
a[14,7] = 1.0
a[23,5] = 1.0
a[34,6] = 1.0
a[35,2] = 1.0
a[38,3] = 1.0
a[42,1] = 1.0
a[70,3] = 1.0
a[74,3] = 1.0
a[105,8] = 1.0
a[112,3] = 1.0
a[115,0] = 1.0
a[127,2] = 1.0
a[130,1] = 1.0
a[148,6] = 1.0
a[182,3] = 1.0
a[184,2] = 1.0
a[194,3] = 1.0
a[209,6] = 1.0
x_pat[0,2] = 1.0
x_pat[2,4] = 1.0
x_pat[2,8] = 1.0
x_pat[3,8] = 1.0
x_pat[4,3] = 1.0
x_pat[8,8] = 1.0
x_pat[9,6] = 1.0
x_pat[10,8] = 1.0
x_pat[12,4] = 1.0
x_pat[14,5] = 1.0
x_pat[17,8] = 1.0
x_pat[21,6] = 1.0
x_pat[23,3] = 1.0
x_pat[25,8] = 1.0
x_pat[26,8] = 1.0
x_pat[28,8] = 1.0
x_pat[31,5] = 1.0
x_pat[31,8] = 1.0
x_pat[39,8] = 1.0
x_pat[40,6] = 1.0
x_pat[41,8] = 1.0
x_pat[42,6] = 1.0
x_pat[43,4] = 1.0
x_pat[46,3] = 1.0
x_pat[48,8] = 1.0
x_pat[50,6] = 1.0
x_pat[51,5] = 1.0
x_pat[54,8] = 1.0
x_pat[57,8] = 1.0
x_pat[58,3] = 1.0
x_pat[58,8] = 1.0
x_pat[59,5] = 1.0
x_pat[60,8] = 1.0
x_pat[61,8] = 1.0
x_pat[62,8] = 1.0
x_pat[63,7] = 1.0
x_pat[64,8] = 1.0
x_pat[65,8] = 1.0
x_pat[66,8] = 1.0
x_pat[67,8] = 1.

In [ ]:
sol

'{ "SolutionInfo": { "Status": 2, "Runtime": 3.2237800002098083e+02, "Work": 3.0677486120578448e+02, "ObjVal": 42100, "ObjBound": 42100, "ObjBoundC": 42100, "MIPGap": 0, "IntVio": 0, "BoundVio": 0, "ConstrVio": 0, "IterCount": 1046742, "BarIterCount": 0, "NLBarIterCount": 0, "PDHGIterCount": 0, "NodeCount": 747, "SolCount": 10, "PoolObjBound": 42100, "PoolNObjVal": [ 42100, 42480, 42970, 43730, 44160, 45010, 46450, 46790, 48310, 55410]}, "Vars": [ { "VarName": "a[0,2]", "X": 1}, { "VarName": "a[12,4]", "X": 1}, { "VarName": "a[14,7]", "X": 1}, { "VarName": "a[23,5]", "X": 1}, { "VarName": "a[34,6]", "X": 1}, { "VarName": "a[35,2]", "X": 1}, { "VarName": "a[38,3]", "X": 1}, { "VarName": "a[42,1]", "X": 1}, { "VarName": "a[70,3]", "X": 1}, { "VarName": "a[74,3]", "X": 1}, { "VarName": "a[105,8]", "X": 1}, { "VarName": "a[112,3]", "X": 1}, { "VarName": "a[115,0]", "X": 1}, { "VarName": "a[127,2]", "X": 1}, { "VarName": "a[130,1]", "X": 1}, { "VarName": "a[148,6]", "X": 1}, { "VarName": "a